[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB01_dicom_hu_windowing.ipynb)

# NB01 · DICOM, Hounsfield units and windowing
**Artificial Intelligence in Medical Imaging** · Day 1 · Lab 1b (22 min) + windowing duel (8 min)

| | |
|---|---|
| Runtime | CPU |
| Data | a synthetic CT slice and a synthetic 3D volume made in the notebook; one small sample DICOM that ships with `pydicom` |
| Checkpoints | air ≈ −1000 HU · window function · NIfTI orientation |
| Tiers | **Core** = TODO cells · **Plus** = write your own tools · **Challenge** = resample a volume |

**Learning goals.** (1) Read the header of a DICOM file. (2) Convert stored values to Hounsfield units. (3) Window an image and say what it hides. (4) Open a 3D volume and find out which way it is oriented.

In [ ]:
%pip install -q "pydicom>=3.0,<4" "nibabel>=5.2,<6"
import numpy as np, matplotlib.pyplot as plt
import pydicom, nibabel as nib
print("pydicom", pydicom.__version__, "| nibabel", nib.__version__)
from pathlib import Path
Path("data").mkdir(exist_ok=True)      # NB01 can run on its own, without NB00
rng = np.random.default_rng(0)

## 1. A CT slice for the whole notebook
We make a synthetic slice so that you know the truth: air is air, water is water, and a small lesion hides in the soft tissue. The code is given; read it, you do not need to edit it.

In [ ]:
def make_ct_phantom(n=256, seed=0):
    """Synthetic chest-like CT slice in Hounsfield units (HU). Not patient data.
    Returns hu (float32) and a dict of masks. A low-contrast lesion (+35 HU) hides in soft tissue,
    and a lung nodule hides in the lung."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)
    cy, cx = n / 2, n / 2
    hu = np.full((n, n), -1000.0, np.float32)                       # air
    body = ((xx - cx) / (0.42 * n)) ** 2 + ((yy - cy) / (0.32 * n)) ** 2 <= 1
    hu[body] = 40.0                                                  # soft tissue
    lungL = ((xx - (cx - 0.17 * n)) / (0.12 * n)) ** 2 + ((yy - cy) / (0.20 * n)) ** 2 <= 1
    lungR = ((xx - (cx + 0.17 * n)) / (0.12 * n)) ** 2 + ((yy - cy) / (0.20 * n)) ** 2 <= 1
    lungs = lungL | lungR
    hu[lungs] = -800.0
    spine = ((xx - cx) ** 2 + (yy - (cy + 0.22 * n)) ** 2) <= (0.04 * n) ** 2
    hu[spine] = 450.0                                                # bone
    lesion = ((xx - cx) ** 2 + (yy - (cy - 0.02 * n)) ** 2) <= (0.035 * n) ** 2
    hu[lesion] = 75.0                                                # soft tissue + 35 HU
    nodule = ((xx - (cx - 0.17 * n)) ** 2 + (yy - (cy - 0.05 * n)) ** 2) <= (0.02 * n) ** 2
    hu[nodule] = -450.0                                              # nodule inside lung
    water = ((xx - (cx + 0.05 * n)) ** 2 + (yy - (cy + 0.12 * n)) ** 2) <= (0.025 * n) ** 2
    hu[water] = 0.0                                                  # water-like cyst
    hu += rng.normal(0, 8, hu.shape).astype(np.float32)             # scanner noise
    return hu, {"body": body, "lungs": lungs, "lesion": lesion, "nodule": nodule, "spine": spine, "water": water}

hu_true, masks = make_ct_phantom()
print("shape:", hu_true.shape, "| HU range:", float(hu_true.min()), "to", float(hu_true.max()))

## 2. Stored values are not Hounsfield units
A DICOM file stores integers. The header tells you how to convert them:

**HU = RescaleSlope × stored + RescaleIntercept**

Below we *simulate* the storage step: the file keeps `uint16` values, slope 1, intercept −1024.

In [ ]:
slope, intercept = 1.0, -1024.0
stored = np.clip(np.round((hu_true - intercept) / slope), 0, 65535).astype(np.uint16)
print("stored dtype:", stored.dtype, "| stored range:", int(stored.min()), "to", int(stored.max()))

**TODO 1.** Convert the stored values back to Hounsfield units. Predict first: what stored value does air (−1000 HU) have?

In [ ]:
# TODO 1: convert `stored` to HU with the formula above.
# Hint: cast to float first (stored is uint16), then multiply and add.
hu = None

### Checkpoint 1 · air ≈ −1000 HU, water ≈ 0 HU

In [ ]:
air = hu[:10, :10].mean()                    # top-left corner is outside the body
water = hu[masks["water"]].mean()
print(f"air = {air:.0f} HU | water = {water:.1f} HU")
assert hu is not None and hu.dtype.kind == "f", "hu must be a float array"
assert -1030 < air < -970, "Air should be near -1000 HU. Did you add the intercept?"
assert abs(water) < 5, "Water should be near 0 HU"
print("Checkpoint 1 passed")

> **Pitfall.** `stored - 1024` on a `uint16` array wraps around instead of going negative. Try it: `stored[0,0] - 1024`. That is why we cast to float first.

In [ ]:
print("uint16 arithmetic:", stored[0, 0] - np.uint16(1024), "<- wrapped around")
print("float arithmetic :", np.float32(stored[0, 0]) - 1024)

## 3. Read a real DICOM header
`pydicom` ships a tiny CT file for testing. Look at the header. If the file cannot be loaded (no internet, old version), skip to section 4; nothing later depends on it.

In [ ]:
from pydicom.data import get_testdata_file
try:
    ds = pydicom.dcmread(get_testdata_file("CT_small.dcm"))
    # TODO 2: print Modality, PixelSpacing, RescaleSlope, RescaleIntercept
    # Hint: the header fields are attributes, e.g. ds.Modality
    print("Modality           :", ...)
    print("Pixel spacing (mm) :", ...)
    print("Rescale slope/int. :", ..., ...)
    print("Image shape/dtype  :", ds.pixel_array.shape, ds.pixel_array.dtype)
except Exception as e:
    ds = None
    print("Could not load the sample file:", repr(e), "-> continue with the next section")

**Privacy check.** Which of these fields could identify a person? List what the file contains. (We come back to this in the afternoon.)

In [ ]:
identifying = ["PatientName", "PatientID", "PatientBirthDate", "InstitutionName", "StudyDate", "ReferringPhysicianName"]
if ds is not None:
    for k in identifying:
        print(f"{k:24s}", repr(ds.get(k, "(absent)")))
else:
    print("No file loaded; skip.")

> **Reflection 2.** If you delete `PatientName`, is the file anonymous? Name two other places where identity can hide (header and pixels).

## 4. Windowing
A screen shows about 256 grey levels, a CT has thousands of HU values. A **window** chooses which range to show: *level* is the centre, *width* is the range mapped from black to white. **The data do not change, only the display.**

**TODO 3.** Write `apply_window(hu, level, width)`: clip to `[level - width/2, level + width/2]` and rescale to 0–255 (`uint8`).

In [ ]:
def apply_window(hu, level, width):
    # TODO 3: 1) compute lo and hi  2) clip hu to [lo, hi]  3) rescale to 0..255  4) return uint8
    ...
    return None

### Checkpoint 2 · the window function

In [ ]:
t = apply_window(np.array([-1000.0, 0.0, 1000.0]), level=0, width=200)
assert t is not None and t.dtype == np.uint8, "return a uint8 array"
assert t[0] == 0 and t[2] == 255 and t[1] in (127, 128), f"unexpected output {t}"
print("Checkpoint 2 passed:", t)

In [ ]:
windows = {"default (wide)": (0, 2000), "soft tissue": (40, 400), "lung": (-600, 1500), "bone": (400, 1800)}
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (name, (L, W)) in zip(ax, windows.items()):
    a.imshow(apply_window(hu, L, W), cmap="gray", vmin=0, vmax=255); a.set_title(f"{name}\nL={L}, W={W}"); a.axis("off")
plt.tight_layout(); plt.show()

## 5. Windowing duel: find the hidden lesion (5 min, pairs)
A lesion is **35 HU brighter** than the soft tissue around it. In the wide window you cannot see it. Find a **level and width** that shows it, write them down, and raise both hands.

Use the sliders (or edit the two numbers). The function `duel_score` tells you if it is visible: the lesion must differ by at least 40 grey levels from its surroundings.

In [ ]:
def duel_score(level, width, target="lesion"):
    img = apply_window(hu, level, width).astype(float)
    m = masks[target]
    from scipy import ndimage as ndi
    ring = ndi.binary_dilation(m, iterations=6) & ~ndi.binary_dilation(m, iterations=2)
    diff = abs(img[m].mean() - img[ring].mean())
    return diff, diff >= 40

def show_window(level=0, width=2000, target="lesion"):
    diff, ok = duel_score(level, width, target)
    plt.figure(figsize=(5, 5)); plt.imshow(apply_window(hu, level, width), cmap="gray", vmin=0, vmax=255)
    plt.title(f"L={level}, W={width} | contrast {diff:.0f} grey levels | {'VISIBLE' if ok else 'not visible'}"); plt.axis("off"); plt.show()

try:
    import ipywidgets as w
    w.interact(show_window, level=w.IntSlider(0, -1000, 1000, 10), width=w.IntSlider(2000, 10, 3000, 10), target=["lesion", "nodule"]);
except Exception:
    show_window(0, 2000)      # edit these two numbers and re-run

**Write down:** level = ____ width = ____ . Now explain in one sentence *why* a narrow window makes a 35 HU difference visible. (Hint: 35 HU out of 2000 HU is how many of 256 grey levels? Out of 120 HU?)

In [ ]:
print("wide window  :", round(35 / 2000 * 255, 1), "grey levels")
print("narrow window:", round(35 / 120 * 255, 1), "grey levels")
assert duel_score(60, 120)[1], "level 60 / width 120 should show the lesion"
assert not duel_score(0, 2000)[1], "the wide default window should hide the lesion"
assert not duel_score(500, 150)[1], "a window far from soft tissue shows nothing"
print("Duel reference checks passed (60/120 visible, 0/2000 and 500/150 not visible)")

> **Plus.** Choose `target="nodule"` and find a window that shows the lung nodule. Which window would a radiologist call a *lung window*?
>
> **Link to AI.** Many models take an 8-bit windowed image as input. The window you pick silently decides what the model *can* see.

## 6. A 3D volume (NIfTI) and its orientation
A CT or MRI is a stack of slices. **NIfTI** stores one array plus an **affine** matrix that says where each voxel sits in space. We build a small volume with 1 × 1 × 3 mm voxels, save it, and read it back. The volume is stored in **LAS** orientation on purpose: this is a classic trap.

In [ ]:
def make_volume(nx=64, ny=64, nz=40):
    x, y, z = np.mgrid[0:nx, 0:ny, 0:nz].astype(np.float32)
    vol = np.zeros((nx, ny, nz), np.float32)
    vol[((x - nx/2)/(0.4*nx))**2 + ((y - ny/2)/(0.3*ny))**2 + ((z - nz/2)/(0.45*nz))**2 <= 1] = 100.0      # body
    vol[((x - 0.75*nx)**2 + (y - ny/2)**2 + (z - nz/2)**2) <= 36] = 300.0                                   # marker, patient LEFT in LAS
    return vol

vol = make_volume()
affine = np.diag([-1.0, 1.0, 3.0, 1.0])          # x runs towards the patient's Left, y Anterior, z Superior
nib.save(nib.Nifti1Image(vol, affine), "data/volume_las.nii.gz")
img = nib.load("data/volume_las.nii.gz")
print("shape :", img.shape, "| voxel size (mm):", img.header.get_zooms())
print("axis codes:", nib.aff2axcodes(img.affine))

**TODO 4.** (a) Compute the physical size of the volume in mm (shape × voxel size). (b) Reorient the image to the standard **RAS** orientation with `nib.as_closest_canonical`.

In [ ]:
zooms = img.header.get_zooms()
# TODO 4a: physical size in mm = shape * voxel size, for each axis
size_mm = None
# TODO 4b: reorient to RAS (hint: nib.as_closest_canonical(img))
img_ras = None
data_ras = np.asarray(img_ras.dataobj) if img_ras is not None else None
print("size (mm):", size_mm)

### Checkpoint 3 · orientation

In [ ]:
raw = np.asarray(img.dataobj)
com = lambda a: np.argwhere(a > 200).mean(axis=0)            # centre of the bright marker
assert size_mm is not None and abs(size_mm[2] - 120.0) < 1e-3, "z size should be 40 slices x 3 mm = 120 mm"
assert nib.aff2axcodes(img_ras.affine) == ("R", "A", "S"), "reorient to RAS"
assert com(raw)[0] > raw.shape[0] / 2, "in the raw LAS array the marker (patient left) is at high x"
assert com(data_ras)[0] < data_ras.shape[0] / 2, "in RAS the patient's left is at low x"
print("Checkpoint 3 passed: the same marker moved to the other side of the array")

In [ ]:
def show_three_planes(v, zooms):
    cx, cy, cz = [s // 2 for s in v.shape]
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(v[:, :, cz].T, origin="lower", cmap="gray", aspect=zooms[1] / zooms[0]); ax[0].set_title("axial")
    ax[1].imshow(v[:, cy, :].T, origin="lower", cmap="gray", aspect=zooms[2] / zooms[0]); ax[1].set_title("coronal")
    ax[2].imshow(v[cx, :, :].T, origin="lower", cmap="gray", aspect=zooms[2] / zooms[1]); ax[2].set_title("sagittal")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
show_three_planes(data_ras, img_ras.header.get_zooms())

> **Reflection 3.** Remove the `aspect=` arguments and look again. What happens to the coronal and sagittal views, and why? What could go wrong if you resized a volume to a cube without checking the voxel size?
>
> **Why it matters.** A silent left-right flip in a training pipeline gives a model that looks fine and puts the wrong side on a report.

### Challenge · resample to isotropic voxels
Resample the volume to 1 × 1 × 1 mm with `scipy.ndimage.zoom`. Check that the new shape is about (64, 64, 120).

In [ ]:
# [CHALLENGE]
from scipy import ndimage as ndi
# TODO (Challenge): zoom factors = voxel size in mm / target voxel size (1 mm). Use order=1.
iso = None
print("isotropic shape:", None if iso is None else iso.shape)
assert iso is not None and iso.shape[0] == 64 and iso.shape[2] in (119, 120, 121)

## Summary
- Stored values are not HU: apply slope and intercept (as float).
- A window changes the display, not the data, and it decides what a model can see.
- A volume is an array plus an affine: check axis codes and voxel size before you trust a view or resize.

*Next:* NB02 (image processing: from scratch to libraries).